# Preprocessing Berita dari Detik.com

In [2]:
# Colab cell 1: mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install Sastrawi pyspellchecker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 71.0 MB/s eta 0:00:00


In [5]:
# Colab cell 2: impor libraries
import os
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from spellchecker import SpellChecker

# Pastikan download resource NLTK
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [9]:
# Colab cell 3: setup stopwords, stemmer, spellchecker custom
stop_words = set(stopwords.words("indonesian"))

factory = StemmerFactory()
stemmer = factory.create_stemmer()

from spellchecker import SpellChecker
spell = SpellChecker(language=None)  # kosong, biar kita isi sendiri

# === Daftar kata baku khas berita ===
custom_words = [
    "presiden","wakil","menteri","gubernur","bupati","polisi","jaksa","hakim",
    "dpr","tni","kpk","korupsi","politik","ekonomi","bisnis","harga","pasar","inflasi",
    "pendidikan","kesehatan","rumah","sakit","pasien","vaksin","covid","hukum",
    "olahraga","pertandingan","skor","gol","liga","medali",
    "internasional","nasional","jakarta","indonesia","detik","berita"
]

spell.word_frequency.load_words(custom_words)

# === Normalisasi manual (gaul → baku) ===
normalize_dict = {
    "gak": "tidak",
    "nggak": "tidak",
    "ngga": "tidak",
    "dr": "dari",
    "yg": "yang",
    "tp": "tetapi",
    "utk": "untuk",
    "biarpun": "meskipun"
}

def normalize_word(word):
    if word in normalize_dict:
        return normalize_dict[word]
    correction = spell.correction(word)
    return correction if correction is not None else word


In [10]:
# Colab cell 4: definisi fungsi preprocessing bertahap
def preprocessing_steps(text):
    original = text

    # Lowercase
    text_lower = original.lower()

    # Remove angka & simbol/tanda baca
    no_symbol = re.sub(r"\d+", " ", text_lower)
    no_symbol = no_symbol.translate(str.maketrans("", "", string.punctuation))

    # Tokenisasi
    tokens = nltk.word_tokenize(no_symbol)

    # Stopword removal
    no_stop = [w for w in tokens if w not in stop_words]

    # Spell correction / pembakuan ejaan
    corrected = [spell.correction(w) if spell.correction(w) is not None else w for w in no_stop]

    # Stemming
    stemmed = [stemmer.stem(w) for w in corrected]

    return {
        "before": original,
        "after_lower_no_symbol": " ".join(tokens),
        "after_stopword": " ".join(no_stop),
        "after_corrected": " ".join(corrected),
        "after_stemmed": " ".join(stemmed),
        "tokens_final": stemmed  # atau bisa pakai corrected / stemmed sesuai kebutuhan
    }

In [12]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
# Colab cell 5: load data & aplikasikan
# Ganti path ke folder-drive kamu
file_path = "/content/drive/MyDrive/Semester 7/berita_detik_baru.csv"
df = pd.read_csv(file_path, dtype=str)  # dtype=str supaya semua teks dianggap string

# Terapkan preprocessing ke kolom 'isi'
results = df["isi"].fillna("").apply(preprocessing_steps)

# Tambahkan kolom hasil ke dataframe
df["before"] = results.apply(lambda x: x["before"])
df["after_lower_no_symbol"] = results.apply(lambda x: x["after_lower_no_symbol"])
df["after_stopword"] = results.apply(lambda x: x["after_stopword"])
df["after_corrected"] = results.apply(lambda x: x["after_corrected"])
df["after_stemmed"] = results.apply(lambda x: x["after_stemmed"])
df["tokens_final"] = results.apply(lambda x: x["tokens_final"])


In [14]:
# Colab cell 6: simpan hasil ke Drive
output_path = "/content/drive/MyDrive/Semester 7/berita_detik_preprocessed.csv"
df.to_csv(output_path, index=False)
print("Hasil preprocessing tersimpan di:", output_path)

Hasil preprocessing tersimpan di: /content/drive/MyDrive/Semester 7/berita_detik_preprocessed.csv


In [17]:
# Tampilkan 5 berita teratas lengkap semua tahapan + tokenisasi
pd.set_option("display.max_colwidth", 200)
display(df[[
    "before",
    "after_lower_no_symbol",
    "after_stopword",
    "after_corrected",
    "after_stemmed",
    "tokens_final"
]].head(5))


,before,after_lower_no_symbol,after_stopword,after_corrected,after_stemmed,tokens_final
0,"Maroko - Dua tahun pascagempa, ribuan warga Maroko masih tinggal di tenda darurat. Mereka menuntut bantuan lebih besar di tengah gencarnya proyek stadion.",maroko dua tahun pascagempa ribuan warga maroko masih tinggal di tenda darurat mereka menuntut bantuan lebih besar di tengah gencarnya proyek stadion,maroko pascagempa ribuan warga maroko tinggal tenda darurat menuntut bantuan gencarnya proyek stadion,maroko pascagempa ribuan harga maroko tinggal tenda darurat menuntut bantuan gencarnya proyek stadion,maroko pascagempa ribu harga maroko tinggal tenda darurat tuntut bantu gencar proyek stadion,"[maroko, pascagempa, ribu, harga, maroko, tinggal, tenda, darurat, tuntut, bantu, gencar, proyek, stadion]"
1,Rekonstruksi kasus Alvi Maulana (24) yang mutilasi Tiara Angelina Saraswati diwarnai kemarahan warga. Alvi mendapat cacian hingga umpatan dari warga saat digelandang menuju lokasi rekonstruksi di ...,rekonstruksi kasus alvi maulana yang mutilasi tiara angelina saraswati diwarnai kemarahan warga alvi mendapat cacian hingga umpatan dari warga saat digelandang menuju lokasi rekonstruksi di kos ka...,rekonstruksi alvi maulana mutilasi tiara angelina saraswati diwarnai kemarahan warga alvi cacian umpatan warga digelandang lokasi rekonstruksi kos kawasan lidah wetan lakarsantri surabaya pantauan...,rekonstruksi alvi maulana mutilasi tiara angelina saraswati diwarnai kemarahan harga alvi cacian umpatan harga digelandang lokasi rekonstruksi gol kawasan liga wetan lakarsantri surabaya pantauan ...,rekonstruksi alvi maulana mutilasi tiara angelina saraswati warna marah harga alvi caci umpat harga gelandang lokasi rekonstruksi gol kawasan liga wetan lakarsantri surabaya pantau detikjatim loka...,"[rekonstruksi, alvi, maulana, mutilasi, tiara, angelina, saraswati, warna, marah, harga, alvi, caci, umpat, harga, gelandang, lokasi, rekonstruksi, gol, kawasan, liga, wetan, lakarsantri, surabaya..."
2,"Kecelakaan lalu lintas melibatkan truk dan dua unit angkot (angkutan kota) di Kampung Tunggilis, Kecamatan Cileungsi, Bogor, Jawa Barat. Tiga orang luka-luka akibat kecelakaan itu. ""Korban kecelak...",kecelakaan lalu lintas melibatkan truk dan dua unit angkot angkutan kota di kampung tunggilis kecamatan cileungsi bogor jawa barat tiga orang lukaluka akibat kecelakaan itu korban kecelakaan luka ...,kecelakaan lintas melibatkan truk unit angkot angkutan kota kampung tunggilis kecamatan cileungsi bogor jawa barat orang lukaluka akibat kecelakaan korban kecelakaan luka ringan orang pengemudi an...,kecelakaan lintas melibatkan truk tni angkot angkutan kota kampung tunggilis kecamatan cileungsi bogor jaksa barat orang lukaluka akibat kecelakaan korban kecelakaan liga ringan orang pengemudi an...,celaka lintas libat truk tni angkot angkut kota kampung tunggilis camat cileungsi bogor jaksa barat orang lukaluka akibat celaka korban celaka liga ringan orang kemudi angkot bernopol f dpr kemudi...,"[celaka, lintas, libat, truk, tni, angkot, angkut, kota, kampung, tunggilis, camat, cileungsi, bogor, jaksa, barat, orang, lukaluka, akibat, celaka, korban, celaka, liga, ringan, orang, kemudi, an..."
3,"Mahkamah Konstitusi (MK) akan menggelar sidang gugatan terhadap UU TNI yang baru disahkan. Ada lima gugatan yang putusannya akan dibacakan hari ini. Dilihat dari situs MK, Rabu (17/5/2025), sidang...",mahkamah konstitusi mk akan menggelar sidang gugatan terhadap uu tni yang baru disahkan ada lima gugatan yang putusannya akan dibacakan hari ini dilihat dari situs mk rabu sidang putusan gugatan u...,mahkamah konstitusi mk menggelar sidang gugatan uu tni disahkan gugatan putusannya dibacakan situs mk rabu sidang putusan gugatan uu nomor perubahan uu nomor tentara nasional indonesia tni digelar...,mahkamah konstitusi kpk menggelar sidang gugatan uu tni disahkan gugatan putusannya dibacakan situs kpk rabu sidang putusan gugatan uu nomor perubahan uu nomor tentara nasional indon